# Mini-teste D-19 + Re-inducao C2 D-21 - Qwen2.5-7B-Instruct-AWQ

**Configuracao obrigatoria antes de salvar e rodar:**
1. Settings -> Accelerator -> **GPU T4 x2**
2. Settings -> Internet -> **On**
3. Settings -> Secrets -> adicione `GITHUB_TOKEN` e `HF_TOKEN` (**marque "Attach to notebook"** para cada um)
4. Para retomar: Add Data -> Your Work -> versao anterior -> **Add as input**
5. Para rodar: Save Version -> **Save & Run All** (nao Run All Cells)

**Tempo estimado:** ~30 min (D-19: 1.600 frases + 200 paralelas workers=16 + re-inducao C2 D-21)

In [ ]:
# Celula 1 - Configuracao
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret('GITHUB_TOKEN')
HF_TOKEN     = secrets.get_secret('HF_TOKEN')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

REPO_SLUG    = 'felipemoraissoares/does-ontology-matter'
REPO_URL     = f'https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git'
BENCH_URL    = 'https://github.com/cenguix/Text2KGBench.git'
BENCH_COMMIT = '50a3d255371b8817cdff70fd88459ac82b339cfe'

WORK_DIR   = '/kaggle/working'
REPO_DIR   = f'{WORK_DIR}/does-ontology-matter'
BENCH_DIR  = f'{WORK_DIR}/Text2KGBench'
RESULT_DIR = f'{REPO_DIR}/resultados/piloto'
D19_DIR    = f'{REPO_DIR}/resultados/d19'
C2_DIR     = f'{REPO_DIR}/configs/c2'
C2_RAW_DIR = f'{REPO_DIR}/resultados/c2_inducao'
VLLM_LOG   = f'{WORK_DIR}/vllm.log'
MODELO     = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
PORT       = 8000

GIT_EMAIL  = 'felipemoraissoares@gmail.com'
GIT_NAME   = 'Felipe Morais (Kaggle)'

print(f'REPO_DIR:  {REPO_DIR}')
print(f'D19_DIR:   {D19_DIR}')
print(f'BENCH_COMMIT: {BENCH_COMMIT[:12]}...')

In [ ]:
# Celula 2 - Helpers
import subprocess

def run(cmd, check=True, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'Falhou (rc={result.returncode}): {cmd[:120]}')
    return result

def git(cmd):
    return run(f'git -C "{REPO_DIR}" {cmd}', check=False)

In [ ]:
# Celula 3 - Clonar repositorios
if not os.path.exists(REPO_DIR):
    run(f'git clone --depth=1 {REPO_URL} {REPO_DIR}')
else:
    print('Repo ja existe - atualizando...')
    git('pull')

if not os.path.exists(BENCH_DIR):
    run(f'git init {BENCH_DIR}')
    run(f'git -C {BENCH_DIR} fetch --depth=1 {BENCH_URL} {BENCH_COMMIT}')
    run(f'git -C {BENCH_DIR} checkout FETCH_HEAD')
    actual = run(f'git -C {BENCH_DIR} rev-parse HEAD', check=False).stdout.strip()
    if actual != BENCH_COMMIT:
        print(f'AVISO: bench commit {actual[:12]} difere do esperado {BENCH_COMMIT[:12]}')
    else:
        print(f'Text2KGBench OK - commit {BENCH_COMMIT[:12]}')
else:
    print('Text2KGBench ja existe.')

git(f'config user.email "{GIT_EMAIL}"')
git(f'config user.name "{GIT_NAME}"')
print('Clones OK.')

In [ ]:
# Celula 4 - Restaurar resultados anteriores (sessao interrompida)
import glob, shutil

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(D19_DIR, exist_ok=True)
os.makedirs(C2_DIR, exist_ok=True)
os.makedirs(C2_RAW_DIR, exist_ok=True)

restaurados = 0
input_base = '/kaggle/input'
if os.path.exists(input_base):
    for dataset_name in os.listdir(input_base):
        for src in glob.glob(f'{input_base}/{dataset_name}/**/piloto_*.jsonl', recursive=True):
            dest = os.path.join(RESULT_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest); restaurados += 1
                print(f'  Restaurado piloto: {os.path.basename(src)}')
        for src in glob.glob(f'{input_base}/{dataset_name}/**/d19_*.jsonl', recursive=True):
            dest = os.path.join(D19_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest); restaurados += 1
                print(f'  Restaurado D19: {os.path.basename(src)}')
        for src in glob.glob(f'{input_base}/{dataset_name}/**/raw_tkg_*.jsonl', recursive=True):
            dest = os.path.join(C2_RAW_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest); restaurados += 1
                print(f'  Restaurado C2 raw: {os.path.basename(src)}')
        for src in glob.glob(f'{input_base}/{dataset_name}/**/calibracao_*.json', recursive=True):
            dest = os.path.join(C2_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest); restaurados += 1
                print(f'  Restaurado C2 calib: {os.path.basename(src)}')

if restaurados > 0:
    print(f'\nTotal restaurado: {restaurados} arquivo(s).')
else:
    print('Nenhum resultado anterior encontrado - execucao completa.')

In [ ]:
# Celula 5 - Instalar dependencias
run('pip install -q vllm openai sentence-transformers scikit-learn nltk scipy numpy pandas statsmodels')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import sys
sys.path.insert(0, f'{REPO_DIR}/src')
print('Dependencias OK.')

In [ ]:
# Celula 6 - Subir vLLM em segundo plano
# dtype=half obrigatorio para T4 (fp16). tensor-parallel-size=2 usa ambas as GPUs T4.

vllm_cmd = (
    f'vllm serve {MODELO} '
    f'--port {PORT} '
    f'--dtype half '
    f'--tensor-parallel-size 2 '
    f'--max-model-len 4096 '
    f'--gpu-memory-utilization 0.90 '
    f'> {VLLM_LOG} 2>&1 &'
)
os.system(vllm_cmd)
print(f'vLLM iniciado em segundo plano. Log: {VLLM_LOG}')

In [ ]:
# Celula 7 - Aguardar vLLM pronto
import time, urllib.request, urllib.error

VLLM_URL  = f'http://localhost:{PORT}'
TIMEOUT   = 360
INTERVALO = 10
elapsed   = 0

while elapsed < TIMEOUT:
    try:
        urllib.request.urlopen(f'{VLLM_URL}/health', timeout=3)
        print(f'vLLM pronto apos {elapsed}s.')
        break
    except (urllib.error.URLError, ConnectionRefusedError, OSError):
        if elapsed % 30 == 0:
            print(f'  aguardando vLLM... ({elapsed}s / {TIMEOUT}s)')
        time.sleep(INTERVALO)
        elapsed += INTERVALO
else:
    print(f'ERRO: vLLM nao respondeu em {TIMEOUT}s. Ultimas linhas do log:')
    run(f'tail -40 {VLLM_LOG}', check=False)
    raise RuntimeError('vLLM nao iniciou. Verifique GPU e modelo.')

In [ ]:
# Celula 8 - Carregar manifesto D-19 com caminhos do Kaggle
import json

os.makedirs(D19_DIR, exist_ok=True)

with open(f'{REPO_DIR}/configs/pipeline/minilancador_d19.json', encoding='utf-8') as f:
    d19_exps = json.load(f)

# Ajusta caminhos de saida: relativo -> absoluto no Kaggle
for exp in d19_exps:
    if 'saida' in exp:
        exp['saida'] = f"{D19_DIR}/{os.path.basename(exp['saida'])}"

d19_path = f'{WORK_DIR}/minilancador_d19_kaggle.json'
with open(d19_path, 'w', encoding='utf-8') as f:
    json.dump(d19_exps, f, ensure_ascii=False, indent=2)

print(f'Manifesto D-19 com {len(d19_exps)} experimentos -> {d19_path}')
for exp in d19_exps:
    existe = '[ja existe]' if os.path.exists(exp['saida']) else '[pendente]'
    print(f"  {exp['id']}  {existe}")

In [ ]:
# Celula 9 - Executar mini-teste D-19
# FORCE_RERUN=True: remove APENAS d19_*.jsonl (preserva piloto-v2 existente).
# Mude para False para retomar uma sessao interrompida.

import glob

FORCE_RERUN = True
if FORCE_RERUN:
    removidos = glob.glob(f'{D19_DIR}/d19_*.jsonl')
    for f_rm in removidos:
        os.remove(f_rm)
    if removidos:
        print(f'FORCE_RERUN=True: {len(removidos)} JSONL d19_* removidos.')
    else:
        print('FORCE_RERUN=True: nenhum JSONL d19_* encontrado (primeira execucao).')

PYPATH = {**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'}

run(
    f'python -m t2kg.run '
    f'--manifesto {d19_path} '
    f'--bench_dir {BENCH_DIR} '
    f'--skip_existentes '
    f'--verbose',
    cwd=REPO_DIR,
    env=PYPATH
)
print('\nMini-teste D-19 concluido.')

In [ ]:
# Celula 10 - Re-inducao C2 com D-21 (threshold=0.70, freq_min=5, raw JSONL)
import json as _json

os.makedirs(C2_RAW_DIR, exist_ok=True)

manifesto_c2 = _json.load(open(f'{REPO_DIR}/configs/pipeline/reinducao_c2_kaggle.json'))
print(f'Re-inducao C2 D-21: {len(manifesto_c2)} tarefa(s), threshold=0.70, freq_min=5')

for tarefa in manifesto_c2:
    onto    = tarefa['onto']
    dataset = tarefa['dataset']
    seed    = tarefa['seed']

    invalidos = [
        f'{C2_DIR}/{dataset}_{onto}_induced.json',
        f'{C2_DIR}/{dataset}_{onto}_frases_seed{seed}.json',
        f'{C2_DIR}/calibracao_{dataset}_{onto}.json',
    ]
    for p in invalidos:
        if os.path.exists(p):
            os.remove(p)
            print(f'  Removido (invalido D-17): {os.path.basename(p)}')

    raw_jsonl = f"{C2_RAW_DIR}/{os.path.basename(tarefa['saida_raw_jsonl'])}"

    print(f"\n--- Induzindo C2 para {dataset}/{onto} (threshold={tarefa['threshold']}, freq_min={tarefa['freq_min']}) ---")
    run(
        f'python {REPO_DIR}/scripts/induzir_c2.py '
        f"--dataset {tarefa['dataset']} "
        f"--onto {tarefa['onto']} "
        f"--url {tarefa['url']} "
        f"--modelo_llm {tarefa['modelo']} "
        f"--threshold {tarefa['threshold']} "
        f"--freq_min {tarefa['freq_min']} "
        f"--n_frases {tarefa['n_frases']} "
        f"--seed {tarefa['seed']} "
        f"{'--calibrar' if tarefa['calibrar'] else ''} "
        f'--saida_raw_jsonl {raw_jsonl} '
        f'--saida_dir {C2_DIR}',
        cwd=REPO_DIR,
        env=PYPATH,
    )

print('\nRe-inducao C2 D-21 concluida.')
git('add -A')
git('commit -m "auto: C2 re-inducao D-21 (threshold=0.70, freq_min=5, raw JSONL)"')
git('push')
print('Push C2 OK.')

In [ ]:
# Celula 10b - Analise automatica D-20: tabela de scores + paralelismo

run(
    f'python {REPO_DIR}/scripts/analisar_resultados_d19.py '
    f'--d19_dir {D19_DIR} '
    f'--piloto_dir {RESULT_DIR} '
    f'--bench_dir {BENCH_DIR} '
    f'--manifesto {d19_path}',
    cwd=REPO_DIR,
    env=PYPATH,
    check=False
)

In [ ]:
# Celula 11 - Push final de todos os resultados
git('add -A')
git('commit -m "auto: mini-teste D-19 completo + C2 D-21"')
git('push')
print('Push final OK.')

In [ ]:
# Celula 12 - Verificar log do vLLM (diagnostico)
run(f'tail -30 {VLLM_LOG}', check=False)